# 06 — Ensemble Analysis, Interpretability & Error Audit

**Explaining model decisions and auditing failure modes**

This notebook dives into *why* the best model (LightGBM) makes its decisions, examines the confidence distribution of its predictions, and audits the most damaging errors — false positives (merging different businesses) and false negatives (missing true duplicates). It also measures inter-model agreement to understand ensemble diversity.

In [ ]:
import sys
sys.path.insert(0, '.')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings("ignore")

from IPython.display import Image, display

try:
    import lightgbm as lgb
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False
    print("[WARNING] lightgbm not installed; Sections 2 will be limited.")

sns.set_theme(style='whitegrid')
%matplotlib inline

# Load all three data sources
from src.features import FEATURE_COLUMNS
feat_df  = pd.read_csv('data/processed/feature_matrix.csv')
oof      = pd.read_csv('results/tables/oof_predictions.csv')
pairs    = pd.read_csv('data/processed/benchmark_pairs.csv')

print(f'feat_df: {feat_df.shape}')
print(f'oof:     {oof.shape}')
print(f'pairs:   {pairs.shape}')
print(f'OOF columns: {list(oof.columns)}')

### 1. LightGBM Feature Importances (Pre-computed)

Feature importances from the trained LightGBM model, saved during the model training pipeline.

In [ ]:
fpath = 'results/plots/feature_importance_lgbm.png'
if os.path.exists(fpath):
    display(Image(fpath))
else:
    print(f'[INFO] {fpath} not found — will generate in the next cell.')

### 2. Manual Feature Importance Bar Chart

We retrain LightGBM on the full feature matrix and extract `feature_importances_` (gain-based). This tells us which features the model relies on most heavily.

In [ ]:
if not HAS_LGBM:
    print('[SKIP] lightgbm not available.')
else:
    X_all = feat_df[FEATURE_COLUMNS].values
    y_all = feat_df['label'].values

    lgb_model = lgb.LGBMClassifier(
        n_estimators=200, learning_rate=0.05, num_leaves=31,
        random_state=42, n_jobs=-1, verbose=-1)
    lgb_model.fit(X_all, y_all)

    importances = pd.Series(lgb_model.feature_importances_, index=FEATURE_COLUMNS)
    importances = importances.sort_values(ascending=True)

    plt.figure(figsize=(10, 6))
    colors = sns.color_palette('viridis', len(importances))
    bars = plt.barh(importances.index, importances.values,
                    color=colors, edgecolor='black', linewidth=0.5)
    plt.xlabel('Feature Importance (Gain)', fontsize=12)
    plt.title('LightGBM Feature Importances (Full-Data Fit)', fontsize=13, fontweight='bold')
    for bar, val in zip(bars, importances.values):
        plt.text(val + importances.max() * 0.01, bar.get_y() + bar.get_height() / 2,
                 f'{val:,.0f}', va='center', fontsize=9)
    plt.tight_layout()
    os.makedirs('results/plots', exist_ok=True)
    plt.savefig('results/plots/feature_importance_lgbm_manual.png', dpi=150, bbox_inches='tight')
    plt.show()

### 3. Out-of-Fold Prediction Confidence Distribution

Plotting the histogram of LightGBM's predicted probabilities, colored by true label, reveals model **calibration**. Well-calibrated: matches cluster near 1.0, non-matches near 0.0. The overlap region is where errors occur.

In [ ]:
if 'LightGBM_prob' not in oof.columns:
    print('[INFO] LightGBM_prob column not found in oof_predictions.csv')
    print('Available columns:', list(oof.columns))
else:
    label_col = 'label' if 'label' in oof.columns else 'y_true'

    fig, ax = plt.subplots(figsize=(10, 5))
    for lbl, lname, color in [(1, 'True Match (label=1)', '#2980b9'),
                               (0, 'True Non-Match (label=0)', '#e74c3c')]:
        subset = oof[oof[label_col] == lbl]['LightGBM_prob']
        ax.hist(subset, bins=50, alpha=0.6, color=color, label=f'{lname} (n={len(subset):,})',
                edgecolor='none')

    ax.axvline(0.5, color='black', linestyle='--', linewidth=1.5, label='Decision threshold (0.5)')
    ax.set_xlabel('Predicted Probability (Match)', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)
    ax.set_title('LightGBM OOF Confidence Distribution', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    plt.tight_layout()
    plt.savefig('results/plots/oof_confidence_distribution.png', dpi=150, bbox_inches='tight')
    plt.show()

### 4. False Positive Error Audit

False positives are the most dangerous errors in entity resolution — they cause incorrect merging of distinct businesses. Here we surface the **most confident** FPs (highest predicted probability despite being true non-matches) for manual inspection.

In [ ]:
if 'LightGBM_prob' not in oof.columns or 'LightGBM_pred' not in oof.columns:
    print('[INFO] LightGBM_prob / LightGBM_pred not found in OOF file.')
else:
    label_col = 'label' if 'label' in oof.columns else 'y_true'

    # Align by row index — no shared ID column
    merged = pd.concat([
        pairs.reset_index(drop=True),
        oof.reset_index(drop=True)
    ], axis=1)

    fp = merged[(merged[label_col] == 0) & (merged['LightGBM_pred'] == 1)].copy()
    fp_sorted = fp.sort_values('LightGBM_prob', ascending=False).head(5)

    show_cols = ['s1_name', 'probe_name', 'pair_type', 'LightGBM_prob']
    show_cols = [c for c in show_cols if c in fp_sorted.columns]

    print(f'Total False Positives: {len(fp):,}')
    fp_sorted[show_cols].style\
        .background_gradient(cmap='Reds', subset=['LightGBM_prob'])\
        .format({'LightGBM_prob': '{:.4f}'})\
        .set_properties(**{'text-align': 'left'})

### 5. False Negative Error Audit

False negatives are missed true matches. While less harmful than FPs in entity resolution, they still leave redundant records in the database. The hardest FNs are pairs that look very different on the surface but are the same real-world business.

In [ ]:
if 'LightGBM_prob' not in oof.columns or 'LightGBM_pred' not in oof.columns:
    print('[INFO] LightGBM_prob / LightGBM_pred not found in OOF file.')
else:
    fn = merged[(merged[label_col] == 1) & (merged['LightGBM_pred'] == 0)].copy()
    fn_sorted = fn.sort_values('LightGBM_prob', ascending=True).head(5)

    show_cols = ['s1_name', 'probe_name', 'pair_type', 'LightGBM_prob']
    show_cols = [c for c in show_cols if c in fn_sorted.columns]

    print(f'Total False Negatives: {len(fn):,}')
    fn_sorted[show_cols].style\
        .background_gradient(cmap='Blues', subset=['LightGBM_prob'])\
        .format({'LightGBM_prob': '{:.4f}'})\
        .set_properties(**{'text-align': 'left'})

### 6. Model Agreement Matrix

Pairwise agreement between all 7 models on OOF predictions. High agreement (>95%) means the two models are redundant — an ensemble of them adds little value. Low agreement (<85%) indicates different error patterns, which makes ensembling beneficial.

In [ ]:
pred_cols = [c for c in oof.columns if c.endswith('_pred')]
print(f'Prediction columns found: {pred_cols}')

if len(pred_cols) < 2:
    print('[INFO] Need at least 2 prediction columns for agreement matrix.')
else:
    n = len(pred_cols)
    agreement = np.zeros((n, n))

    for i, c1 in enumerate(pred_cols):
        for j, c2 in enumerate(pred_cols):
            agreement[i, j] = (oof[c1] == oof[c2]).mean()

    short_names = [c.replace('_pred', '').replace('_', ' ') for c in pred_cols]
    agr_df = pd.DataFrame(agreement, index=short_names, columns=short_names)

    plt.figure(figsize=(10, 8))
    mask = np.eye(n, dtype=bool)  # hide diagonal
    sns.heatmap(agr_df, annot=True, fmt='.3f', cmap='YlOrRd_r',
                vmin=0.75, vmax=1.0, linewidths=0.5,
                annot_kws={'size': 10}, mask=mask)
    plt.xticks(rotation=35, ha='right', fontsize=9)
    plt.yticks(fontsize=9)
    plt.title('Pairwise Model Agreement (% rows where both predict the same label)',
              fontsize=12, fontweight='bold', pad=12)
    plt.tight_layout()
    plt.savefig('results/plots/model_agreement_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Print most / least agreeing pairs
    pairs_agree = []
    for i in range(n):
        for j in range(i+1, n):
            pairs_agree.append((short_names[i], short_names[j], agreement[i,j]))
    pairs_agree.sort(key=lambda x: x[2])
    print('\n5 Most Diverse Pairs (lowest agreement):')
    for m1, m2, ag in pairs_agree[:5]:
        print(f'  {m1} vs {m2}: {ag:.3f}')
    print('\n5 Most Similar Pairs (highest agreement):')
    for m1, m2, ag in pairs_agree[-5:][::-1]:
        print(f'  {m1} vs {m2}: {ag:.3f}')